# Population

Total population by county across stepped years, compared to the regional totals and to legacy LUVit estimates when available.

In [1]:
import sys
from pathlib import Path

# Make summary_scripts/ importable regardless of where the kernel started.
_HERE = Path.cwd()
for candidate in [_HERE, _HERE / "summary_scripts", _HERE.parent / "summary_scripts"]:
    if (candidate / "validation_data_input.py").exists():
        sys.path.insert(0, str(candidate))
        break

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from validation_data_input import (
    load_config, load_settings, load_unrolled, load_unrolled_regional,
    load_targets, control_id_lookup, load_legacy_pop, load_legacy_units,
    available_years,
)
from util import (
    COUNTY_ID_TO_NAME, COUNTY_ORDER, COUNTY_COLORS, INDICATORS,
    apply_plotly_theme, format_int, pct_diff, style_diff_table, passfail_badge,
)

cfg = load_config()
settings = load_settings()
BASE_YEAR = settings.get("base_year")
END_YEAR = settings.get("end_year")
TARGETS_END_YEAR = settings.get("targets_end_year")

ctrl = load_unrolled().merge(control_id_lookup(), on="control_id", how="left")
ctrl["county"] = ctrl["county_id"].map(COUNTY_ID_TO_NAME)


## County totals at base year and end year

In [2]:
by_county_year = (
    ctrl.groupby(['county', 'year'])['total_pop'].sum().unstack('year')
)
by_county_year = by_county_year.reindex(COUNTY_ORDER)
by_county_year['Region'] = by_county_year.sum(axis=0)  # column-wise won't help
by_county_year = by_county_year.drop(columns='Region')
summary = pd.DataFrame({
    f'{BASE_YEAR}': by_county_year[BASE_YEAR],
    f'{TARGETS_END_YEAR}': by_county_year[TARGETS_END_YEAR],
    f'{END_YEAR}': by_county_year[END_YEAR],
    'Growth (base→targets)': by_county_year[TARGETS_END_YEAR] - by_county_year[BASE_YEAR],
    '% Growth': (by_county_year[TARGETS_END_YEAR] / by_county_year[BASE_YEAR] - 1) * 100,
})
summary.loc['Region'] = summary.sum(numeric_only=True)
summary.loc['Region', '% Growth'] = (
    summary.loc['Region', f'{TARGETS_END_YEAR}']
    / summary.loc['Region', f'{BASE_YEAR}'] - 1
) * 100
style_diff_table(
    summary,
    int_cols=[f'{BASE_YEAR}', f'{TARGETS_END_YEAR}', f'{END_YEAR}', 'Growth (base→targets)'],
    pct_cols=['% Growth'], pct_threshold=100,
)

,2023,2044,2050,Growth (base→targets),% Growth
county,,,,,
King,"2,331,162","2,881,068","3,038,186","549,906",+23.6%
Kitsap,"275,524","338,390","356,352","62,866",+22.8%
Pierce,"921,372","1,170,859","1,242,139","249,487",+27.1%
Snohomish,"855,882","1,133,358","1,212,638","277,476",+32.4%
Region,"4,383,940","5,523,675","5,849,315","1,139,735",+26.0%


## Trajectory by county

In [3]:
trend = ctrl.groupby(['county', 'year'])['total_pop'].sum().reset_index()
fig = px.line(
    trend, x='year', y='total_pop', color='county',
    category_orders={'county': COUNTY_ORDER},
    color_discrete_map=COUNTY_COLORS, markers=True,
    labels={'total_pop': 'Population', 'year': 'Year', 'county': 'County'},
    title='Total population by county',
)
apply_plotly_theme(fig)

## Regional total — controls vs. legacy LUVit

In [4]:
reg = load_unrolled_regional()[['year', 'total_pop']].copy()
reg['source'] = 'New pipeline'
legacy = load_legacy_pop()
frames = [reg]
if legacy is not None:
    leg = legacy.rename(columns={c: c.lower() for c in legacy.columns})
    if 'year' in leg.columns and 'total_pop' in leg.columns:
        leg = leg[['year', 'total_pop']].copy()
        leg['source'] = 'Legacy LUVit'
        frames.append(leg)
compare = pd.concat(frames, ignore_index=True)
fig = px.line(
    compare, x='year', y='total_pop', color='source', markers=True,
    title='Regional population — pipeline vs. legacy',
    labels={'total_pop': 'Population', 'year': 'Year', 'source': 'Source'},
)
if legacy is None:
    fig.add_annotation(
        text='Legacy LUVit estimates not available for this run.',
        xref='paper', yref='paper', x=0.5, y=1.08,
        showarrow=False, font=dict(color='#888'),
    )
apply_plotly_theme(fig)

## Notes

- Base year, targets year, and end year come from `configs/settings.yaml` of the active example.
- Legacy comparison loads `examples/legacy_luvit/output/total_pop_estimates_all_years.csv` if present and non-empty.